# Competition Simulation 3: Blending + Stacking with OOF Discipline

This notebook demonstrates an ensemble workflow that stays leaderboard-safe:
1. Train diverse base models and generate strict OOF predictions.
2. Build weighted blending and stacking on top of OOF outputs.
3. Discuss **shake risk** (public leaderboard noise vs private reality).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from sklearn.base import clone
from sklearn.datasets import make_friedman1
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

## 1) Create a synthetic competition split

No manual downloads: we generate data with nonlinear structure plus a mild train→leaderboard shift.


In [ ]:
n_total = 3000
n_train = 2200

X_raw, y_raw = make_friedman1(
    n_samples=n_total,
    n_features=10,
    noise=1.2,
    random_state=RANDOM_SEED,
)

feature_names = [f"f{i}" for i in range(X_raw.shape[1])]
df = pd.DataFrame(X_raw, columns=feature_names)

time_idx = np.arange(n_total)
df["time_idx"] = time_idx
df["seasonal"] = np.sin(time_idx / 35)

# Mild distribution shift in the leaderboard period
df.loc[n_train:, "f0"] = df.loc[n_train:, "f0"] + 0.35
df.loc[n_train:, "f3"] = df.loc[n_train:, "f3"] * 1.15

y = (
    y_raw
    + 0.8 * df["seasonal"].to_numpy()
    + 0.45 * (df["f0"] * df["f1"]).to_numpy()
    + rng.normal(0, 0.4, n_total)
)

X_train = df.iloc[:n_train].reset_index(drop=True)
y_train = y[:n_train]

X_lb = df.iloc[n_train:].reset_index(drop=True)
y_lb = y[n_train:]

public_size = int(0.35 * len(X_lb))

print(f"Train rows: {len(X_train)}, leaderboard rows: {len(X_lb)}, public rows: {public_size}")
X_train.head()

## 2) Train base models and collect OOF predictions

OOF rule: each training row must be predicted only by models that never trained on that row.
This is the foundation for leakage-free blending/stacking.


In [ ]:
base_models = {
    "ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.5)),
    "random_forest": RandomForestRegressor(
        n_estimators=180,
        max_depth=8,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
    "gbrt": GradientBoostingRegressor(
        n_estimators=260,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.85,
        random_state=RANDOM_SEED,
    ),
    "extra_trees": ExtraTreesRegressor(
        n_estimators=220,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

def make_oof_and_eval_predictions(models, X_fit, y_fit, X_eval, splitter):
    oof_df = pd.DataFrame(index=np.arange(len(X_fit)))
    eval_df = pd.DataFrame(index=np.arange(len(X_eval)))
    model_rows = []

    for name, model in models.items():
        oof_pred = np.zeros(len(X_fit))
        eval_fold_preds = []

        for tr_idx, va_idx in splitter.split(X_fit, y_fit):
            m = clone(model)
            m.fit(X_fit.iloc[tr_idx], y_fit[tr_idx])

            oof_pred[va_idx] = m.predict(X_fit.iloc[va_idx])
            eval_fold_preds.append(m.predict(X_eval))

        oof_df[name] = oof_pred
        eval_df[name] = np.mean(eval_fold_preds, axis=0)
        model_rows.append({"model": name, "oof_rmse": rmse(y_fit, oof_pred)})

    summary = pd.DataFrame(model_rows).sort_values("oof_rmse").reset_index(drop=True)
    return oof_df, eval_df, summary

oof_base, lb_base_preds, base_summary = make_oof_and_eval_predictions(
    base_models, X_train, y_train, X_lb, cv
)
base_summary

## 3) Weighted blending (OOF-tuned)

We optimize blend weights on OOF predictions only (never on leaderboard labels).


In [ ]:
top_models = base_summary["model"].head(3).tolist()
weight_grid = np.linspace(0.0, 1.0, 21)

best_blend = None
for w1, w2 in product(weight_grid, repeat=2):
    w3 = 1.0 - w1 - w2
    if w3 < 0 or w3 > 1:
        continue
    weights = np.array([w1, w2, w3])
    pred = oof_base[top_models].to_numpy() @ weights
    score = rmse(y_train, pred)
    if best_blend is None or score < best_blend["oof_rmse"]:
        best_blend = {"weights": weights, "oof_rmse": score}

blend_oof = oof_base[top_models].to_numpy() @ best_blend["weights"]
blend_lb = lb_base_preds[top_models].to_numpy() @ best_blend["weights"]

print("Top models for blend:", top_models)
print("Best blend weights:", np.round(best_blend["weights"], 3))
print(f"Blend OOF RMSE: {best_blend['oof_rmse']:.4f}")

## 4) Stacking (meta-model trained on OOF features)

We build stack features from base-model OOF predictions and train a ridge meta-model.
Meta-level OOF is also generated to keep estimates realistic.


In [ ]:
meta_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED + 7)
stack_oof = np.zeros(len(y_train))

for tr_idx, va_idx in meta_cv.split(oof_base, y_train):
    meta_model = Ridge(alpha=1.0, positive=True)
    meta_model.fit(oof_base.iloc[tr_idx], y_train[tr_idx])
    stack_oof[va_idx] = meta_model.predict(oof_base.iloc[va_idx])

stack_oof_rmse = rmse(y_train, stack_oof)

final_meta_model = Ridge(alpha=1.0, positive=True)
final_meta_model.fit(oof_base, y_train)
stack_lb = final_meta_model.predict(lb_base_preds)

ensemble_summary = pd.DataFrame(
    [
        {"model": base_summary.iloc[0]["model"], "oof_rmse": float(base_summary.iloc[0]["oof_rmse"])},
        {"model": "weighted_blend", "oof_rmse": float(best_blend["oof_rmse"])},
        {"model": "ridge_stack", "oof_rmse": float(stack_oof_rmse)},
    ]
).sort_values("oof_rmse")
ensemble_summary

## 5) Shake-risk analysis: public vs private ranking

We compute one public/private split and then run repeated random splits to estimate how unstable leaderboard ranks can be.


In [ ]:
candidate_lb_preds = {name: lb_base_preds[name].to_numpy() for name in base_summary["model"]}
candidate_lb_preds["weighted_blend"] = blend_lb
candidate_lb_preds["ridge_stack"] = stack_lb

def leaderboard_table(pred_dict, y_full, public_n):
    rows = []
    for name, pred in pred_dict.items():
        rows.append(
            {
                "model": name,
                "public_rmse": rmse(y_full[:public_n], pred[:public_n]),
                "private_rmse": rmse(y_full[public_n:], pred[public_n:]),
            }
        )

    table = pd.DataFrame(rows)
    table["public_rank"] = table["public_rmse"].rank(method="min")
    table["private_rank"] = table["private_rmse"].rank(method="min")
    table["rank_shift"] = (table["public_rank"] - table["private_rank"]).abs()
    return table.sort_values("public_rmse").reset_index(drop=True)

lb_once = leaderboard_table(candidate_lb_preds, y_lb, public_size)
lb_once

In [ ]:
shake_records = []
n_sim = 120
all_idx = np.arange(len(y_lb))

for sim in range(n_sim):
    public_idx = rng.choice(all_idx, size=public_size, replace=False)
    private_mask = np.ones(len(y_lb), dtype=bool)
    private_mask[public_idx] = False

    sim_rows = []
    for name, pred in candidate_lb_preds.items():
        sim_rows.append(
            {
                "model": name,
                "public_rmse": rmse(y_lb[public_idx], pred[public_idx]),
                "private_rmse": rmse(y_lb[private_mask], pred[private_mask]),
            }
        )

    sim_df = pd.DataFrame(sim_rows)
    sim_df["public_rank"] = sim_df["public_rmse"].rank(method="min")
    sim_df["private_rank"] = sim_df["private_rmse"].rank(method="min")
    sim_df["abs_rank_shift"] = (sim_df["public_rank"] - sim_df["private_rank"]).abs()
    sim_df["sim"] = sim
    shake_records.append(sim_df)

shake_df = pd.concat(shake_records, ignore_index=True)

shake_summary = (
    shake_df.groupby("model")
    .agg(
        avg_public_rmse=("public_rmse", "mean"),
        avg_private_rmse=("private_rmse", "mean"),
        avg_abs_rank_shift=("abs_rank_shift", "mean"),
        p90_abs_rank_shift=("abs_rank_shift", lambda s: np.percentile(s, 90)),
    )
    .sort_values(["avg_abs_rank_shift", "avg_private_rmse"])
)

shake_summary

In [ ]:
plot_models = [base_summary.iloc[0]["model"], "weighted_blend", "ridge_stack"]
fig, ax = plt.subplots(figsize=(7, 4))
for model_name in plot_models:
    ax.hist(
        shake_df.loc[shake_df["model"] == model_name, "abs_rank_shift"],
        bins=np.arange(-0.5, 6.5, 1.0),
        alpha=0.45,
        label=model_name,
    )

ax.set_title("Leaderboard shake-risk (abs rank shift)")
ax.set_xlabel("Absolute rank shift: public vs private")
ax.set_ylabel("Simulation count")
ax.legend()
plt.tight_layout()

### Ensembling takeaway

- OOF discipline is mandatory: blends/stacks trained on in-fold predictions are leakage-prone.
- Blending often gives robust gains with low complexity.
- Public leaderboard improvements can still be noisy, so use shake-risk checks before over-trusting small deltas.
